# LLM 10: Streaming & Incremental Rendering

Hands-on:
1. Stream tokens from Claude and measure TTFT
2. Stream JSON from a tool call and parse it progressively
3. Implement mid-stream cancellation
4. Simulate lost-connection recovery
5. Exercises: streaming UI sketch, token-budget enforcement

Deps: `pip install anthropic openai partial-json-parser`

## 1. Basic Token Streaming + TTFT

In [ ]:
import os, time

prompt = 'Write a three-sentence story about a lighthouse keeper.'

if os.getenv('ANTHROPIC_API_KEY'):
    from anthropic import Anthropic
    client = Anthropic()
    t0 = time.perf_counter()
    ttft = None
    printed = 0
    with client.messages.stream(
        model='claude-sonnet-4-6',
        max_tokens=300,
        messages=[{'role':'user','content':prompt}],
    ) as stream:
        for text in stream.text_stream:
            if ttft is None:
                ttft = time.perf_counter() - t0
            print(text, end='', flush=True)
            printed += len(text)
    total = time.perf_counter() - t0
    print(f'\n\nTTFT: {ttft*1000:.0f} ms   total: {total:.2f} s   chars: {printed}')
else:
    print('Skipped (no ANTHROPIC_API_KEY).')

## 2. Stream Typed Events (Text, Tool Use, Thinking)

The raw event stream tells you exactly what kind of block is being generated.

In [ ]:
if os.getenv('ANTHROPIC_API_KEY'):
    from anthropic import Anthropic
    client = Anthropic()
    with client.messages.stream(
        model='claude-sonnet-4-6',
        max_tokens=200,
        messages=[{'role':'user','content':'Say hi in three words.'}],
    ) as stream:
        for event in stream:
            print(event.type, end=' | ')
else:
    print('Skipped.')

## 3. Streaming JSON with Partial Parsing

Use tool use to force structured output and parse the JSON as it arrives.

In [ ]:
try:
    from partial_json_parser import loads as partial_loads
except ImportError:
    def partial_loads(s):
        import json
        # crude fallback: try to complete open brackets
        s = s.strip()
        if not s:
            return {}
        for extra in ['', '}', '"}', ']}', '"]}', '}]']:
            try:
                return json.loads(s + extra)
            except Exception:
                continue
        return {}

if os.getenv('ANTHROPIC_API_KEY'):
    from anthropic import Anthropic
    client = Anthropic()
    tool = {
        'name': 'emit_person',
        'description': 'Emit a fictional person record',
        'input_schema': {
            'type': 'object',
            'properties': {
                'name':   {'type': 'string'},
                'age':    {'type': 'integer'},
                'city':   {'type': 'string'},
                'hobbies':{'type':'array','items':{'type':'string'}},
            },
            'required': ['name','age','city','hobbies'],
        },
    }
    buf = ''
    with client.messages.stream(
        model='claude-sonnet-4-6',
        max_tokens=300,
        tools=[tool],
        tool_choice={'type':'tool','name':'emit_person'},
        messages=[{'role':'user','content':'Invent a fictional 40-year-old.'}],
    ) as stream:
        for event in stream:
            if getattr(event, 'type', None) == 'content_block_delta':
                d = event.delta
                if getattr(d, 'type', None) == 'input_json_delta':
                    buf += d.partial_json
                    progressive = partial_loads(buf)
                    print(f'parsed so far: {progressive}')
else:
    print('Skipped.')

## 4. Mid-Stream Cancellation

In [ ]:
import threading, time

if os.getenv('ANTHROPIC_API_KEY'):
    from anthropic import Anthropic
    client = Anthropic()
    cancel = threading.Event()
    # simulate a user clicking "stop" after 0.5 s
    threading.Timer(0.5, cancel.set).start()

    with client.messages.stream(
        model='claude-sonnet-4-6',
        max_tokens=1000,
        messages=[{'role':'user','content':'Write a 500-word poem about the sea.'}],
    ) as stream:
        printed = 0
        for text in stream.text_stream:
            if cancel.is_set():
                stream.close()
                print(f'\n[cancelled after {printed} chars]')
                break
            printed += len(text)
            print(text, end='', flush=True)
else:
    print('Skipped.')

## 5. Exercise: Progressive Form Rendering

Build a CLI that:
1. Streams a tool call that emits a `{name, email, phone, address}` record.
2. After each partial parse, re-renders a form-like view:
```
Name:    Alice█
Email:   ⏳
Phone:   ⏳
Address: ⏳
```
Use ANSI escape sequences to overwrite the view in-place.

This is the shape of forward-looking LLM UIs (Vercel AI SDK Generative UI, Claude's artifacts).

## 6. Exercise: Token-Budget Enforcement

During streaming, keep a running total of output tokens. If it exceeds a per-user budget, cancel the stream and return the partial output plus a budget-exceeded message.

```python
budget = 500
used = 0
for event in stream:
    if event.type == 'message_delta':
        used = event.message.usage.output_tokens
        if used > budget:
            stream.close(); break
```

This is a real pattern for protecting unit economics on user-visible features.

## 7. Takeaways

- **Stream anything that takes >1 s.** TTFT dominates perceived latency.
- **Parse partial JSON** with a proper library for streaming structured output.
- **Cancellation is a feature.** Wire it end-to-end.
- **Disable buffering** at every proxy hop.
- **Stream + cache together** for lowest TTFT on repeat-heavy prefixes.

Next: **11 — LLM Evaluation (Intro)**, where we learn to measure the quality of all this output.